In [1]:
from pathlib import Path

import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

from QuantNado import BamStore
from QuantNado.downstream import (
    annotate_intervals,
    extract_feature_ranges,
    extract_metadata,
    extract_promoters,
    feature_counts,
    load_gtf,
    plot_pca_scatter,
    plot_pca_scree,
    reduce_byranges_signal,
    run_pca,
)


In [2]:
fig_dir = Path("./figures")
fig_dir.mkdir(exist_ok=True)

# Load Dataset

In [3]:
ds = BamStore.open("dataset_full", backend="zarr")
ds

2025-12-20 01:34:57.686 | INFO     | QuantNado.bam_store:open:372 - Opening zarr store at: dataset_full.zarr


<xarray.Dataset> Size: 148GB
Dimensions:        (sample: 20, position_flat: 3088286401, contig: 25)
Coordinates:
  * sample         (sample) int64 160B 0 1 2 3 4 5 6 7 ... 13 14 15 16 17 18 19
  * position_flat  (position_flat) int64 25GB 0 1 2 ... 3088286399 3088286400
  * contig         (contig) int64 200B 0 1 2 3 4 5 6 7 ... 18 19 20 21 22 23 24
    contig_length  (contig) int64 200B dask.array<chunksize=(25,), meta=np.ndarray>
    contig_offset  (contig) int64 200B dask.array<chunksize=(25,), meta=np.ndarray>
Data variables:
    signal         (sample, position_flat) uint16 124GB dask.array<chunksize=(1, 64000), meta=np.ndarray>
Attributes: (12/17)
    assay_by_sample:         ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP',...
    metadata_control:        ['', '', 'Input', 'Input', 'Input', 'Input', 'In...
    metadata_ip:             ['', '', 'H3K27Ac', 'H3K27Ac', 'MLL', 'MLL', 'Me...
    metadata_replicate:      ['', '', '', '', '', '', '', '', '', '', '', '',...
    metadata_scaling_group:  ['default', 'default', 'default', 'default', 'de...
    metadata_timepoint:      ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr',...
    ...                      ...
    assays:                  ATAC,ChIP,RNA
    sample_names:            ['SEM-DMSO', 'SEM-MENi', 'SEM-DMSO-H3K27Ac_H3K27...
    contig_names:            ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6',...
    structure:               ragged (sample × position_flat with contig offsets)
    bin_size:                1
    average_sparsity:        71.70%

# Explore Zarr Dataset



In [4]:
print(ds.attrs["sample_names"])
print(ds.attrs["assay_by_sample"])

['SEM-DMSO', 'SEM-MENi', 'SEM-DMSO-H3K27Ac_H3K27ac', 'SEM-DMSO-H3K27Ac_Input', 'SEM-DMSO-MLL_Input', 'SEM-DMSO-MLL_MLL', 'SEM-DMSO-Menin_Input', 'SEM-DMSO-Menin_Menin', 'SEM-MENi-H3K27Ac_H3K27ac', 'SEM-MENi-H3K27Ac_Input', 'SEM-MENi-MLL_Input', 'SEM-MENi-MLL_MLL', 'SEM-MENi-Menin_Input', 'SEM-MENi-Menin_Menin', 'SEM-DMSO-rep1', 'SEM-DMSO-rep2', 'SEM-DMSO-rep3', 'SEM-MENi-rep1', 'SEM-MENi-rep2', 'SEM-MENi-rep3']
['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'RNA', 'RNA', 'RNA', 'RNA', 'RNA', 'RNA']


In [5]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nAttributes:", ds.attrs)
print("\nData variables:", list(ds.data_vars))

Dimensions: FrozenMappingWarningOnValuesAccess({'sample': 20, 'position_flat': 3088286401, 'contig': 25})

Coordinates: ['position_flat', 'contig_length', 'sample', 'contig', 'contig_offset']

Attributes: {'assay_by_sample': ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'RNA', 'RNA', 'RNA', 'RNA', 'RNA', 'RNA'], 'metadata_control': ['', '', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', 'Input', '', '', '', '', '', ''], 'metadata_ip': ['', '', 'H3K27Ac', 'H3K27Ac', 'MLL', 'MLL', 'Menin', 'Menin', 'H3K27Ac', 'H3K27Ac', 'MLL', 'MLL', 'Menin', 'Menin', '', '', '', '', '', ''], 'metadata_replicate': ['', '', '', '', '', '', '', '', '', '', '', '', '', '', 'rep1', 'rep2', 'rep3', 'rep1', 'rep2', 'rep3'], 'metadata_scaling_group': ['default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'default', 'd

In [6]:
# Helpers for ragged coordinates and metadata
metadata_df = extract_metadata(ds)

# Reduce by BED file

In [7]:
# promoters = "/Users/catherine/work/project/QuantNado/data/hg38/promoters_1024bp.bed"

# promoter_ds = reduce_byranges_signal(ds["signal"], bed_file=promoters, reduction="mean")
# promoter_ds

## PCA analysis

Perform Principal Component Analysis on the promoter dataset to visualize sample relationships:

In [8]:
# # Ensure numeric dtype and explicit chunking to avoid auto on object dtypes
# promoter_mean = promoter_ds["mean"].astype("float32")
# promoter_signal = promoter_mean.chunk({"ranges": 10000})

# pca_object, pca_result = run_pca(
#     input_array=promoter_signal,
#     n_components=2,
#     nan_handling_strategy="drop",
#     standardize=True,
#     random_state=42,
#     subset_size=1_000,
#     subset_strategy="random",
#     svd_solver="randomized",
# )

# plot_pca_scree(
#     pca_object=pca_object,
#     filepath=f"{fig_dir}/pca_scree.png",
# )

# plot_pca_scatter(
#     pca_object,
#     pca_result,
#     xaxis_pc=1,
#     yaxis_pc=2,
#     metadata_df=metadata_df,
#     colour_by="assay",
#     shape_by="treatment",
#     sample_column="sample_id",
#     filepath=f"{fig_dir}/pca_plot.png",
# )

# Extract Feature counts

In [9]:
gtf_file = "/Users/catherine/work/project/QuantNado/data/hg38/hg38.ncbiRefSeq.gtf"
gtf = load_gtf(gtf_file)
print(gtf.head())

  seqname     feature  start    end strand gene_id gene_name transcript_id  \
0    chrM  transcript  15956  16023      -    TRNP      TRNP      rna-TRNP   
1    chrM        exon  15956  16023      -    TRNP      TRNP      rna-TRNP   
2    chrM  transcript  15888  15953      +    TRNT      TRNT      rna-TRNT   
3    chrM        exon  15888  15953      +    TRNT      TRNT      rna-TRNT   
4    chrM  transcript  14747  15887      +    CYTB      CYTB      rna-CYTB   

  gene_type gene_biotype  
0      None         None  
1      None         None  
2      None         None  
3      None         None  
4      None         None  


In [ ]:
counts_rna, feature_meta = feature_counts(
    dataset=ds,
    gtf_file=gtf_file,
    feature_type="transcript",
    assay="RNA",
    integerize=True,
    filter_zero=True,
)
display(feature_meta.head())

In [ ]:
# Align RNA counts and metadata
metadata_rna = metadata_df.loc[metadata_df["assay"] == "RNA"].copy()
metadata_rna.index = metadata_rna["sample"].astype(str)
# Keep only columns with any non-null or non-empty values
nonempty_cols = metadata_rna.columns[
    (metadata_rna.notna().any()) | ((metadata_rna != "").any())
]
metadata_rna = metadata_rna[nonempty_cols]
display(metadata_rna.head())

In [ ]:
counts_df = counts_rna.T

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_rna,
    design="~treatment",
)


dds.fit_size_factors()
dds.fit_genewise_dispersions()


dds.obs["size_factors"]
dds.var["genewise_dispersions"]

In [ ]:
# counts_exon, fm_exon = feature_counts(
#     dataset=ds,
#     gtf_file=gtf_file,
#     feature_type="exon",
#     aggregate_by=None,
#     assay="RNA",
#     integerize=True,
#     filter_zero=False,
# )
# print("Exon sums head:", counts_exon.sum().head())
# print("Feature_meta gene_id present?", "gene_id" in fm_exon.columns)
# grouped = counts_exon.groupby(fm_exon["gene_id"]).sum()
# print("Grouped sums head:", grouped.sum().head())
# # Quick diagnostics: are RNA samples present and do they have non-zero signal?
# sample_names_attr = ds.attrs.get("sample_names")
# if sample_names_attr is not None:
#     sample_names = [str(s) for s in sample_names_attr]
# else:
#     sample_coord = ds.coords.get("sample", None)
#     sample_names = [str(s) for s in sample_coord.values] if sample_coord is not None else []

# assay_by_sample = list(ds.attrs.get("assay_by_sample", []))
# rna_samples = [s for s, a in zip(sample_names, assay_by_sample) if a == "RNA"]
# print("RNA samples from attrs:", rna_samples)

# sample_coord = ds.coords.get("sample", None)
# if sample_coord is None:
#     print("Dataset has no 'sample' coordinate.")
# else:
#     coord_samples = [str(s) for s in sample_coord.values]
#     available = [s for s in rna_samples if s in coord_samples]
#     missing = [s for s in rna_samples if s not in coord_samples]
#     print("Samples in coord (first 10):", coord_samples[:10], "..." if len(coord_samples) > 10 else "")
#     print("RNA samples available in coord:", available)
#     if missing:
#         print("Missing RNA samples (not in coord):", missing)
#     if available:
#         rna_signal = ds["signal"].sel(sample=available)
#         subset = rna_signal.isel(ranges=slice(0, min(1000, rna_signal.sizes.get("ranges", 0))))
#         print("Subset shape:", subset.shape)
#         print("Subset sum:", subset.sum().values)
#         print("Subset max:", subset.max().values)
#     else:
#         print("No RNA samples available on coordinate; cannot check signal.")